# Decision Trees - Interpretable Models with Sharp Edges

<hr>

<center>
<div>
<img src="https://raw.githubusercontent.com/davi-moreira/2026Summer_predictive_analytics_purdue_MGMT474/main/notebooks/figures/mgmt_474_ai_logo_02-modified.png" width="200"/>
</div>
</center>

# <center><a class="tocSkip"></center>
# <center>MGMT47400 Predictive Analytics</center>
# <center>Professor: Davi Moreira </center>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/davi-moreira/2026Summer_predictive_analytics_purdue_MGMT474/blob/main/notebooks/11_decision_trees_student.ipynb)

---

## Learning Objectives

By the end of this notebook, you will be able to:

1. Fit decision trees for regression/classification
2. Control complexity (depth, min samples) to manage overfitting
3. Interpret tree structure and failure modes
4. Compare tree vs linear/logistic baselines under CV
5. Handle class imbalance with `class_weight='balanced'` — the first-resort tool, without resorting to SMOTE
6. Document "when a tree is the right tool"

---

> **📋 Participation Reminder:** This notebook contains **2 PAUSE-AND-DO exercises**. You are expected to complete all exercises before submitting your notebook.

---

## 💼 Why This Matters: A Model the Doctor Can Read

The **State Health Department** screening tool works, but the oncologists push back: *"I can't explain to a patient why the model flagged them. I need to see the decision logic."* Logistic regression gives coefficients, but the interaction between features is hidden behind a linear equation.

Decision trees solve this: "If worst radius > 16.8 AND mean texture > 21.4, then malignant." The decision path is a flowchart that any clinician can follow — transparent, auditable, and explainable to patients.

> **Today's focus:** Building decision trees for the breast cancer dataset, visualizing the decision logic, and understanding the overfitting risk of deep, unpruned trees.

> **A question that often comes up here:** *"If the logistic regression from NB06 already scores 0.97 on this dataset, why do we need a different model at all?"* Because "performance" and "explainability" are different goods. A logistic regression weighs every one of the 30 cell-nucleus features through a linear equation; a clinician cannot easily read that equation back to a patient as a sequence of yes/no questions. A decision tree trades a small amount of stability for a decision path that an oncologist can read out loud: *"If `worst_radius` is above 16.8 AND `worst_concave_points` is above 0.14, we flag for biopsy."* That kind of auditable logic is what the advisory board needs before deployment — even if the raw AUC numbers end up similar.

---


In [ ]:
# Setup
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer, fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import roc_auc_score, mean_squared_error, r2_score
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.precision', 4)
RANDOM_SEED = 474
np.random.seed(RANDOM_SEED)
print("✓ Setup complete!")

**Reading the output:**

The setup cell loads the tools for building and visualizing decision trees on the Health Department's breast cancer dataset. `DecisionTreeClassifier` and `DecisionTreeRegressor` implement the CART algorithm — the same recursive-splitting logic that a clinician follows when triaging cases by asking a sequence of yes/no questions. `plot_tree` renders the learned tree structure as a graphic that oncologists can audit before approving the model for patient-facing use. `StratifiedKFold` ensures each cross-validation fold preserves the roughly 63% benign / 37% malignant class balance, preventing folds where malignant cases are underrepresented.

The message `Setup complete!` with **RANDOM_SEED = 474** confirms that every stochastic operation — data splits, tree construction — will produce identical results each time. Reproducibility matters here: if the screening team reruns this notebook next quarter, they must get the same tree structure and the same performance numbers to maintain regulatory trust.

**Key takeaway:** A successful setup cell is the foundation of reproducibility. If any import fails, none of the downstream tree-building code will run in Colab.

---

## 1. Decision Tree Intuition

### How Trees Make Decisions

The oncologists on the Health Department's advisory panel have a specific request: *"Show me why the model flagged this patient."* Decision trees answer that question directly. The CART algorithm (Classification and Regression Trees) builds a flowchart of yes/no questions about cell measurements:

1. Start with all 569 tissue samples at the root
2. Find the single feature + threshold that best separates malignant from benign — "best" means the split that maximizes information gain (classification) or minimizes MSE (regression)
3. Create two child nodes and repeat recursively until a stopping criterion is met

**Example Decision Path (breast cancer screening):**
```
Is worst_radius > 16.8?
  ├─ No: Is worst_concave_points > 0.14?
  │   ├─ No: Predict benign → routine follow-up
  │   └─ Yes: Predict malignant → biopsy referral
  └─ Yes: Predict malignant → biopsy referral
```

### Key Hyperparameters

**Complexity control (all about preventing memorization):**
- `max_depth`: Maximum tree depth — a depth-3 tree has at most 8 leaf nodes, auditable by any clinician
- `min_samples_split`: Minimum samples to split a node — prevents splits based on 2-3 edge cases
- `min_samples_leaf`: Minimum samples in a leaf — ensures each prediction is backed by enough patients
- `max_features`: Number of features to consider per split — limits overfitting to correlated measurements

The root-to-leaf path is a rule the oncologist can follow: "If `worst_radius` > 16.8 AND `worst_concave_points` > 0.14, flag as suspicious." This transparency is exactly what the Health Department's clinical advisory board requires — a model whose reasoning can be explained to the patient sitting across the desk.

## 2. Classification Tree Example

We build the Health Department's first decision tree on the **Wisconsin Breast Cancer** dataset (569 tissue samples, 30 numeric cell-nucleus measurements). Each sample is labeled malignant or benign — the same binary classification task that drives the statewide screening tool. The tree will learn axis-aligned splits on features such as `worst_radius`, `mean_texture`, and `worst_concave_points`.

We cap `max_depth=3` so the resulting tree is small enough for a clinician to review the full decision path in under a minute. Even a shallow tree can achieve surprisingly high accuracy on well-separated classes, and the visualization makes it clear exactly which cell measurements and thresholds drive each screening decision.

> 💡 **Gemini Prompt:** "Load the breast cancer dataset, split 70/30 with stratification (seed 474), fit a DecisionTreeClassifier(max_depth=3), and print train/test accuracy with the overfitting gap."
>
> **After running, verify:**
> - Train/test reflect a 70/30 split
> - Both accuracies and gap are printed
> - Tree uses max_depth=3 and random_state=474
> - All numerical outputs use standard decimal format — no scientific notation


In [ ]:
# Load data
data = load_breast_cancer(as_frame=True)
X = data.data
y = data.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=RANDOM_SEED, stratify=y)

print(f"Train: {len(X_train)} | Test: {len(X_test)}")
print(f"Features: {X.shape[1]}")

# Fit a simple tree
tree_clf = DecisionTreeClassifier(max_depth=3, random_state=RANDOM_SEED)
tree_clf.fit(X_train, y_train)

train_score = tree_clf.score(X_train, y_train)
test_score = tree_clf.score(X_test, y_test)

print(f"\n=== DECISION TREE (max_depth=3) ===")
print(f"Train accuracy: {train_score:.4f}")
print(f"Test accuracy: {test_score:.4f}")
print(f"Overfit gap: {train_score - test_score:.4f}")

**Reading the output:**

The dataset splits into **398 training** and **171 test** tissue samples across **30 cell-nucleus features** — measurements from digitized images of fine-needle aspirates. With `max_depth=3` the tree is constrained to at most 8 leaf nodes, each representing a distinct diagnostic path.

Training accuracy around **0.97-0.98** tells us the shallow tree captures most of the malignant-vs-benign signal using just three levels of splits on features like `worst_radius` and `worst_concave_points`. Test accuracy close behind (gap typically < 0.02) confirms the model generalizes to unseen tissue samples rather than memorizing training patients. The **overfit gap** at the bottom should be small and positive; a negative gap would indicate unusual luck on the test split.

**Why this matters:** At `max_depth=3`, the tree stays shallow enough for the Health Department's oncologists to review every decision path before the model touches a real patient. Deeper trees may gain marginal accuracy but become black boxes — and in a screening deployment where a flagged result triggers a biopsy referral, the clinician needs to see *why*.

> **A question that often comes up here:** *"Why does a shallow tree (depth 3) do almost as well as a deep tree?"* Because the breast cancer dataset has strong, well-separated signals in a handful of cell-nucleus features — `worst_radius`, `worst_concave_points`, and `mean_texture` together carry most of the predictive power. A depth-3 tree uses exactly the right number of splits to exploit those signals without burning its budget on noise. That is a happy feature of this dataset; on messier tabular data the depth at which test accuracy peaks can be much higher (7–10 is common), which is why the next section runs the depth sweep.

---


> 💡 **Gemini Prompt:** "Visualize the fitted decision tree using plot_tree with feature names, class names, filled colors, and rounded boxes on a (20,10) figure."
>
> **After running, verify:**
> - Tree shows feature names at each split, not generic X[i]
> - Nodes are color-filled (darker=more pure)
> - Figure is large enough (20x10) to read all labels
> - All numerical outputs use standard decimal format — no scientific notation


In [ ]:
# Visualize the tree
plt.figure(figsize=(20, 10))
plot_tree(
    tree_clf,
    feature_names=X.columns,
    class_names=data.target_names,
    filled=True,
    rounded=True,
    fontsize=10
)
plt.title("Decision Tree Visualization (max_depth=3)")
plt.tight_layout()
plt.show()

print("💡 Follow a path from root to leaf to see decision logic")
print("💡 Darker colors = more samples, purity of class")

**Reading the output:**

The graphic shows every internal decision node and leaf node of the depth-3 tree — the exact flowchart an oncologist would follow when triaging a tissue sample. Each box contains four pieces of information: the **split rule** (e.g., `worst_radius <= 16.8`), the **Gini impurity** (0 = perfectly pure, meaning all samples in that node share the same diagnosis), the **number of samples** reaching that node, and the **class distribution** (malignant vs. benign counts).

Color intensity encodes purity: darker blue means almost all samples are benign, darker orange means almost all are malignant. Follow any root-to-leaf path and you read a plain-English screening rule: "If `worst_radius` > 16.8 and `worst_concave_points` > 0.14, predict malignant — refer for biopsy."

Notice that the root split uses `worst_radius` — the largest cell nucleus radius in the tissue sample. This makes clinical sense: malignant tumors are characterized by larger, more irregularly shaped nuclei. The tree has independently discovered a pattern that pathologists have known for decades, which builds the oncologist's trust in the model's logic.

**Key takeaway:** This visualization is the primary advantage of decision trees over black-box models. The Health Department can hand this diagram to any clinician at a partner hospital, and they can audit every prediction path before the screening tool goes live.

---

## 3. The Overfitting Problem

### Trees Without Constraints = Memorization

An unrestricted decision tree will keep splitting until every leaf contains a single tissue sample, achieving **100% training accuracy** by memorizing each patient. Deploy that tree at a new partner hospital and its performance collapses — those hyper-specific rules (e.g., "if `mean_symmetry` is between 0.1811 and 0.1813, predict benign") capture noise from one lab's equipment, not genuine malignancy signals.

The depth sweep below makes this tradeoff visible: training accuracy climbs monotonically with depth, while test accuracy peaks at a moderate depth and then declines. Monitoring the **overfit gap** (train accuracy minus test accuracy) is the fastest way to diagnose the problem. A gap near zero means the model generalizes across hospitals; a large gap means the tree has memorized artifacts of the training data.

> 💡 **Gemini Prompt:** "Loop over max_depth=[1,2,3,5,10,20,None], fit a DecisionTreeClassifier for each, collect train/test accuracy and leaf count. Plot train vs test accuracy and overfitting gap across depths."
>
> **After running, verify:**
> - Results DataFrame has max_depth, train_acc, test_acc, gap, n_leaves
> - First subplot shows Train and Test lines diverging with depth
> - Overfitting gap grows with unlimited depth
> - All numerical outputs use standard decimal format — no scientific notation


In [ ]:
# Compare different depths
depths = [1, 2, 3, 5, 10, 20, None]  # None = unlimited
results = []

for depth in depths:
    tree = DecisionTreeClassifier(max_depth=depth, random_state=RANDOM_SEED)
    tree.fit(X_train, y_train)
    
    train_acc = tree.score(X_train, y_train)
    test_acc = tree.score(X_test, y_test)
    
    results.append({
        'max_depth': str(depth),
        'train_acc': train_acc,
        'test_acc': test_acc,
        'gap': train_acc - test_acc,
        'n_leaves': tree.get_n_leaves()
    })

results_df = pd.DataFrame(results)
print("=== DEPTH SWEEP ===")
print(results_df.to_string(index=False))

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy vs depth
axes[0].plot(range(len(depths)), results_df['train_acc'], marker='o', label='Train', linewidth=2)
axes[0].plot(range(len(depths)), results_df['test_acc'], marker='s', label='Test', linewidth=2)
axes[0].set_xticks(range(len(depths)))
axes[0].set_xticklabels(results_df['max_depth'])
axes[0].set_xlabel('Max Depth')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Accuracy vs Tree Depth')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Overfit gap
axes[1].bar(range(len(depths)), results_df['gap'], alpha=0.7, edgecolor='black')
axes[1].set_xticks(range(len(depths)))
axes[1].set_xticklabels(results_df['max_depth'])
axes[1].set_xlabel('Max Depth')
axes[1].set_ylabel('Train - Test Gap')
axes[1].set_title('Overfitting vs Tree Depth')
axes[1].axhline(y=0, color='r', linestyle='--')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 Unrestricted trees achieve 100% training accuracy (pure overfitting!)")
print("💡 Best test performance is at moderate depth")

**Reading the output:**

The results table and two-panel plot reveal the classic bias-variance tradeoff applied to the screening task. In the left panel, training accuracy (blue) climbs steadily and reaches **1.0** for `max_depth=None` (unlimited), while test accuracy (orange) peaks around depth 3-5 then flattens or declines. The right panel shows the overfit gap growing from near zero at depth 1 to a substantial value at unlimited depth.

Key numbers to compare: at `max_depth=3` the gap is typically around **0.01-0.02**, meaning the tree generalizes well to unseen tissue samples. At `max_depth=None` the gap can exceed **0.05-0.10**, with the tree growing to dozens of leaves that memorize training noise — individual patient quirks rather than genuine malignancy patterns.

The `n_leaves` column reinforces this: a depth-3 tree has at most 8 leaves (8 distinct diagnostic paths), while an unrestricted tree may have 20+ leaves on 398 training samples, averaging fewer than 20 patients per leaf. A screening rule backed by only 5 patients is not one the Health Department can defend to its clinical advisory board.

**Why this matters:** This depth sweep is the single most important diagnostic for decision trees. The Health Department needs a model that generalizes across all partner hospitals and lab equipment, not one that memorizes the training hospital's specific patient population. Always run this sweep before selecting a final depth.

> **A question that often comes up here:** *"If unlimited depth always gets 100% training accuracy, why is that a bad thing?"* Because training accuracy measures memorization, not generalization. An unrestricted tree with 398 leaves (one per training patient) has perfectly memorized who is who — but when a new patient arrives whose feature vector does not exactly match any training row, the tree has to guess. The validation and test accuracies in the right panel show what that guessing costs in real deployment. The professional posture: **never ship a model whose training accuracy is 1.00 unless you are 100% sure that is not memorization.**

---


## 📝 PAUSE-AND-DO Exercise 1 (5 minutes)

**Task:** Run a depth sweep and choose depth based on CV.

---

> 💡 **Gemini Prompt:** "Cross-validate DecisionTreeClassifier over depths [2,3,4,5,6,7,8,10,15] using 5-fold StratifiedKFold with ROC-AUC. Print results, identify best depth, and plot CV score with error bars."
>
> **After running, verify:**
> - CV results show mean and std ROC-AUC per depth
> - Best depth marked with vertical line
> - Error bars reflect std across folds
> - All numerical outputs use standard decimal format — no scientific notation


In [ ]:
# YOUR SOLUTION CODE HERE
# Hint: Use the Gemini prompt above for step-by-step guidance
#
# Starter template below — extend it with the plot and richer interpretation.

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
depths_to_test = [2, 3, 4, 5, 6, 7, 8, 10, 15]

cv_results = []
for depth in depths_to_test:
    tree = DecisionTreeClassifier(max_depth=depth, random_state=RANDOM_SEED)
    scores = cross_val_score(tree, X_train, y_train, cv=cv, scoring='roc_auc')
    cv_results.append({
        'max_depth': depth,
        'cv_mean': scores.mean(),
        'cv_std': scores.std()
    })

cv_df = pd.DataFrame(cv_results)
best_depth = cv_df.loc[cv_df['cv_mean'].idxmax(), 'max_depth']

# TODO: Print the CV sweep results
# TODO: Create a line plot with error bars and highlight the best depth

# Line plot with error bars (scaffold — extend styling/annotations as you like)
print(cv_df.to_string(index=False))
print(f"\n✓ Best depth: {best_depth}")

plt.figure(figsize=(10, 5))
plt.errorbar(cv_df['max_depth'], cv_df['cv_mean'], yerr=cv_df['cv_std'],
             marker='o', capsize=5, linewidth=2, color='steelblue')
plt.axvline(x=best_depth, color='red', linestyle='--', label=f'Best depth = {best_depth}')
plt.xlabel('Max Depth')
plt.ylabel('CV ROC-AUC')
plt.title('Decision Tree: CV Depth Selection', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Tree vs Linear Model Comparison

The Health Department now faces a concrete question: does the decision tree's ability to model non-linear feature interactions actually improve screening over the logistic regression baseline from earlier notebooks? Logistic regression fits a single linear decision boundary — it works well when malignant and benign classes are roughly linearly separable in the 30-dimensional feature space. The decision tree carves the space into axis-aligned rectangles and can capture interactions (e.g., "high `worst_radius` AND high `mean_concavity` together signal malignancy") without manual feature engineering.

The comparison below uses the same `StratifiedKFold` cross-validation object for both models so the evaluation is fair — same folds, same patient partitions. We report ROC-AUC on validation folds and on the held-out test set. On the breast cancer dataset the two approaches often perform similarly because the classes are well separated; the real differences emerge on noisier datasets with complex feature interactions.

> 💡 **Gemini Prompt:** "Compare the best-depth DecisionTree vs StandardScaler+LogisticRegression using 5-fold CV ROC-AUC and test ROC-AUC. Print comparison table."
>
> **After running, verify:**
> - Two models compared: Decision Tree and Logistic Regression
> - Both CV and test ROC-AUC reported
> - Comparison DataFrame has Model, CV_Mean, CV_Std, Test_Score
> - Bar plot clearly visualizes the comparison across models/groups
> - All numerical outputs use standard decimal format — no scientific notation


In [ ]:
# Compare tree vs logistic regression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

models = {
    'Decision Tree (best)': DecisionTreeClassifier(max_depth=int(best_depth), random_state=RANDOM_SEED),
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(random_state=RANDOM_SEED, max_iter=1000))
    ])
}

comparison = []
for name, model in models.items():
    scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='roc_auc')
    
    # Fit on full train for test evaluation
    model.fit(X_train, y_train)
    test_score = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
    
    comparison.append({
        'Model': name,
        'CV_Mean': scores.mean(),
        'CV_Std': scores.std(),
        'Test_Score': test_score
    })

comp_df = pd.DataFrame(comparison)
print("=== TREE VS LINEAR COMPARISON ===")
print(comp_df.to_string(index=False))

print("\n💡 Which performs better depends on data structure")
print("💡 Trees handle non-linear relationships naturally")
print("💡 Linear models need manual feature engineering")
# --- Bar plot: Tree vs Logistic comparison ---
comp_df = pd.DataFrame(comparison)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, metric in zip(axes, ['CV_Mean', 'Test_Score']):
    comp_df.plot.bar(x='Model', y=metric, ax=ax, legend=False, color=['steelblue', 'coral'])
    ax.set_title(metric.replace('_', ' '), fontsize=13, fontweight='bold')
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=15)
    for container in ax.containers:
        ax.bar_label(container, fmt='%.4f', fontsize=9)
fig.suptitle('Decision Tree vs Logistic Regression', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


**Reading the output:**

The comparison table shows CV ROC-AUC (mean and standard deviation) plus test-set ROC-AUC for both the tuned decision tree and logistic regression. On the breast cancer dataset, both models typically achieve ROC-AUC above **0.97** — the 30 cell-nucleus features provide strong separability between malignant and benign tissue.

If the tree's CV standard deviation is noticeably larger than logistic regression's, that illustrates the **high-variance** nature of trees: small changes in which patients land in each fold can shift the tree structure significantly (a different root split cascades through the entire tree), while logistic regression produces a stable linear boundary every time. For the Health Department, this stability matters — a model that performs 0.99 on one hospital's data and 0.93 on another is harder to trust than one that consistently delivers 0.97.

The test scores should be consistent with the CV means. If either model's test score falls outside the range mean +/- 2*std, that is a red flag worth investigating (possible data leakage or an unusual patient mix in the test split).

**Why this matters:** This head-to-head comparison establishes whether a tree's interpretability advantage comes with a performance cost. On the screening dataset, logistic regression's stability may make it the better production choice — unless the clinical advisory board insists on the visual decision-path transparency that only a tree provides.

> **A question that often comes up here:** *"How do I decide whether to deploy the tree or the logistic regression when the ROC-AUC numbers are close?"* By shifting the decision from *performance* to *stakeholder needs*. If the clinical advisory board wants an auditable decision path, the tree wins — the interpretability premium is worth the small performance cost. If the board cares only about screening accuracy and stability across hospitals, logistic regression's lower standard deviation across folds tips the balance. Either answer is defensible *as long as you state the tradeoff you are making*. "The tree is better" is not a stakeholder-ready sentence; "The tree is more explainable at the cost of slightly higher variance across folds" is.

---


## 📝 PAUSE-AND-DO Exercise 2 (5 minutes)

**Task:** Write 3 observed tree failure modes (with evidence).

---

### YOUR ANALYSIS: Tree Failure Modes

**Failure Mode 1: Overfitting**  
[Evidence from depth sweep - what happened with unlimited depth?]

**Failure Mode 2: Instability**  
[Evidence from CV std - how much do scores vary across folds?]

**Failure Mode 3: Extrapolation**  
[Trees can only predict values seen in training - what's the implication?]

---

## 5. Regression Trees

Everything we learned about classification trees for the screening task applies to regression: the algorithm still makes recursive binary splits, but instead of Gini impurity it minimizes **mean squared error (MSE)** within each region. Leaf predictions are the average target value of training samples in that leaf — not a class vote, but a continuous number.

We demonstrate with the **California Housing** dataset (20,640 census tracts, 8 features) to show that the overfitting pattern is universal: it is not specific to the Health Department's 569-sample classification problem. Predicting median house value in units of $100k, the depth sweep below reveals the same overfitting signature — an unrestricted regression tree memorizes training tracts while failing on unseen neighborhoods.

> 💡 **Gemini Prompt:** "Load California Housing regression dataset, split 70/30, sweep DecisionTreeRegressor over max_depth=[2,3,5,7,10,15,None], recording train and test R-squared and gap for each."
>
> **After running, verify:**
> - Dataset is California Housing (regression)
> - Results show train/test R-squared and gap per depth
> - Unlimited depth (None) shows \~1.0 train R-squared (severe overfitting)
> - Bar plot clearly visualizes the comparison across models/groups
> - All numerical outputs use standard decimal format — no scientific notation


In [ ]:
# Load regression dataset
california = fetch_california_housing(as_frame=True)
X_reg = california.data
y_reg = california.target

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.3, random_state=RANDOM_SEED
)

# Depth sweep for regression
depths_reg = [2, 3, 5, 7, 10, 15, None]
reg_results = []

for depth in depths_reg:
    tree_reg = DecisionTreeRegressor(max_depth=depth, random_state=RANDOM_SEED)
    tree_reg.fit(X_train_reg, y_train_reg)
    
    train_r2 = tree_reg.score(X_train_reg, y_train_reg)
    test_r2 = tree_reg.score(X_test_reg, y_test_reg)
    
    reg_results.append({
        'max_depth': str(depth),
        'train_r2': train_r2,
        'test_r2': test_r2,
        'gap': train_r2 - test_r2
    })

reg_df = pd.DataFrame(reg_results)
print("=== REGRESSION TREE DEPTH SWEEP ===")
print(reg_df.to_string(index=False))

print("\n💡 Same overfitting pattern as classification")
print("💡 Unrestricted tree memorizes training data")
# --- Bar plot: Regression tree depth comparison ---
reg_df = pd.DataFrame(reg_results)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
x_labels = [str(d) if d is not None else 'None' for d in reg_df['max_depth']]
for ax, metric, label in zip(axes, ['train_r2', 'test_r2'], ['Train R²', 'Test R²']):
    ax.bar(x_labels, reg_df[metric], color='steelblue' if 'Train' in label else 'coral')
    ax.set_xlabel('max_depth')
    ax.set_ylabel('R²')
    ax.set_title(label, fontsize=13, fontweight='bold')
    for i, v in enumerate(reg_df[metric]):
        ax.text(i, v + 0.01, f'{v:.3f}', ha='center', fontsize=8)
fig.suptitle('Regression Tree: Depth vs Performance', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


**Reading the output:**

The regression depth sweep mirrors the classification pattern from the screening task. At `max_depth=2` the tree underfits — too few splits to capture the relationship between features like `MedInc` (median income) and house value. As depth increases, training R-squared approaches **1.0** (the tree memorizes the 14,448 training census tracts), while test R-squared peaks around depth **7-10** and then declines.

Typical values: the best test R-squared is around **0.60-0.70**, meaning a single regression tree explains roughly 60-70% of variance in California median house values. The overfit gap at `max_depth=None` is dramatic, often exceeding **0.30** — far worse than anything we saw in the classification task, because the regression tree has more degrees of freedom to memorize continuous target values.

Compare these numbers to a simple Ridge regression (R-squared around 0.60). A moderately deep tree matches or slightly beats the linear baseline, but an unrestricted tree dramatically overfits. This motivates the ensemble methods we build next: Random Forests (averaging many trees) and Gradient Boosting (sequentially correcting errors) both achieve higher R-squared without memorization.

**Key takeaway:** Whether you are classifying tumors or predicting house prices, the lesson is the same: always limit tree depth and validate on held-out data. The next notebook shows how Random Forests solve the instability problem by averaging hundreds of trees.

> **A question that often comes up here:** *"Why does the regression tree overfit so much more dramatically than the classification tree?"* Because regression trees have more degrees of freedom. A classification leaf outputs a class label (0 or 1); a regression leaf outputs a continuous mean — any number the tree wants. With enough depth, the tree can assign a unique mean to every leaf, effectively memorizing every training tract's target value exactly. Classification cannot go that far, because there are only two possible leaf predictions. This is also why ensemble methods (Random Forests in NB12, Gradient Boosting in NB13) help regression problems *more* dramatically than they help classification — there is simply more variance for the ensemble to average out.

---


## 6. Imbalanced Classes — `class_weight='balanced'` as the First-Resort Tool

The Breast Cancer dataset is only mildly imbalanced (~37% malignant). Many real classification problems are far more skewed: fraud detection (1% positive), customer churn (5–15%), rare-disease screening (under 1%). When the positive class is a small minority, a decision tree's greedy-split algorithm can effectively ignore it — the splits that maximize purity favor the majority class, and the minority class falls through the cracks.

The professional first response is **not** SMOTE, not oversampling, not undersampling. It is a single argument in the estimator constructor: `class_weight='balanced'`. This tells the tree to weight each class inversely to its frequency, so the splits that help the minority class are worth proportionally more. No synthetic data, no risk of data leakage, no controversial resampling.

To make the effect visible on the Breast Cancer dataset, the cell below **artificially down-samples** the malignant cases to about 10% of training — simulating what the model would see on a rare-disease screening task. Then it fits two trees: one with default weights, one with `class_weight='balanced'`.

> 💡 **Gemini Prompt:** "Down-sample the malignant cases in `X_train` and `y_train` so the positive class drops to roughly 10% (keep all benign cases, randomly keep only 20% of malignant cases with `RANDOM_SEED = 474`). Fit two `DecisionTreeClassifier(max_depth=5, random_state=RANDOM_SEED)` models — one with default weights, one with `class_weight='balanced'`. For each, predict on the original `X_test` and print the classification report (use `sklearn.metrics.classification_report` with `target_names=['benign','malignant']`). Compare the recall of the malignant class between the two models."
>
> **After running, verify:**
> - The imbalanced training set has roughly 90% benign and 10% malignant rows
> - The default-weights tree tends to miss malignant cases (low recall for the malignant class)
> - The `class_weight='balanced'` tree recovers visibly higher malignant recall

In [ ]:
# --- Artificially imbalance the training set to simulate a rare-disease screening task ---
from sklearn.metrics import classification_report

rng_imbal = np.random.default_rng(RANDOM_SEED)
train_mask_benign = (y_train == 1)  # 1 = benign in sklearn's breast cancer encoding
train_mask_malig = (y_train == 0)

keep_malig_idx = rng_imbal.choice(
    np.where(train_mask_malig)[0],
    size=max(1, int(0.20 * train_mask_malig.sum())),
    replace=False
)
keep_idx = np.concatenate([np.where(train_mask_benign)[0], keep_malig_idx])
X_train_imbal = X_train.iloc[keep_idx]
y_train_imbal = y_train.iloc[keep_idx]

print(f'Imbalanced training set: {len(y_train_imbal)} rows')
print(f'  Benign    (y=1): {(y_train_imbal == 1).sum()}  ({(y_train_imbal == 1).mean():.1%})')
print(f'  Malignant (y=0): {(y_train_imbal == 0).sum()}  ({(y_train_imbal == 0).mean():.1%})')

# --- Default weights tree ---
tree_default = DecisionTreeClassifier(max_depth=5, random_state=RANDOM_SEED)
tree_default.fit(X_train_imbal, y_train_imbal)
y_pred_default = tree_default.predict(X_test)
print('\n--- Default tree (class_weight=None) ---')
print(classification_report(y_test, y_pred_default,
                            target_names=['malignant', 'benign']))

# --- Balanced weights tree ---
tree_balanced = DecisionTreeClassifier(max_depth=5, class_weight='balanced',
                                       random_state=RANDOM_SEED)
tree_balanced.fit(X_train_imbal, y_train_imbal)
y_pred_balanced = tree_balanced.predict(X_test)
print('\n--- Balanced tree (class_weight="balanced") ---')
print(classification_report(y_test, y_pred_balanced,
                            target_names=['malignant', 'benign']))

**Reading the output:**

Two classification reports side by side. The default tree fit on the imbalanced training set typically shows **lower recall for malignant cases** than the balanced tree — it has learned "most patients are benign" and predicts benign by default. On a real screening tool, that behavior is catastrophic: the cost of missing a malignant case (a false negative) is far higher than the cost of flagging a benign case for follow-up.

`class_weight='balanced'` shifts the recall trade-off without touching the data. sklearn computes `n_samples / (n_classes * np.bincount(y))` internally and multiplies each sample's contribution to the loss by its class weight — minority-class errors cost more, so the tree's greedy splits chase them.

**Anti-SMOTE warning — read this twice.** SMOTE and related oversampling techniques synthesize fake minority-class rows by interpolating between existing ones. This is a *data* solution to a *loss function* problem. Three reasons to avoid it as a first resort:

1. **Synthetic samples are not real data.** Interpolating between two malignant cases does not create a third malignant patient; it creates a point in feature space that no real patient ever occupied. Models trained on synthetic rows can be confidently wrong on real ones.
2. **Leakage risk.** SMOTE must be applied *inside* the cross-validation pipeline or it pollutes the validation fold with synthetic neighbors of its own rows. Subtle, easy to get wrong.
3. **It rarely beats `class_weight`.** Empirically, on most tabular classification tasks, the class-weight lever matches or beats SMOTE and is one line of code.

**Use SMOTE only after** `class_weight='balanced'` has been tried, threshold tuning (NB07) has been tried, and a principled cost-sensitive evaluation still shows the minority class under-served. Not before.

> **A question that often comes up here:** *"If `class_weight='balanced'` is this easy and this effective, why don't we apply it by default on every classification problem?"* Because on balanced or mildly-imbalanced datasets (like the real MedScreen data at 37/63), `class_weight='balanced'` makes almost no difference — the default already weighs the classes appropriately. It only matters when the imbalance is severe enough (say, 5/95 or worse) that the default-weights splits ignore the minority class. Applying it indiscriminately does no harm, but it is not free either: it can slow training on large datasets and can sometimes hurt ROC-AUC slightly on well-balanced data. The professional rule: apply it when imbalance hurts minority-class recall, leave it off otherwise.

---


## 7. When to Use Decision Trees

### Strengths
- ✓ **Interpretable** — the oncologist can read the decision path and explain it to a patient
- ✓ Handle non-linear relationships naturally (e.g., `worst_radius` interacting with `mean_concavity`)
- ✓ No feature scaling needed — splits are based on thresholds, not distances
- ✓ Handle mixed data types (numeric + categorical)
- ✓ Capture interactions automatically without manual feature engineering
- ✓ Fast to train and predict — important for real-time screening at partner hospitals

### Weaknesses
- ✗ **High variance** — small changes in which patients are in the training set produce a completely different tree
- ✗ Easy to overfit — an unrestricted tree memorizes every training patient
- ✗ Poor extrapolation — cannot predict values outside the training range (a regression tree trained on values up to $500k cannot predict $600k)
- ✗ Biased toward features with many unique values
- ✗ Step-function boundaries — staircase approximations of smooth decision surfaces

### Use Decision Trees When:
1. The clinical advisory board requires an auditable, visual decision path
2. You need a quick interpretable baseline before investing in ensemble methods
3. Relationships are highly non-linear with feature interactions
4. The tree will serve as the building block for Random Forests or Gradient Boosting

### Avoid Decision Trees When:
1. High-dimensional sparse data (too many candidate splits, unstable)
2. Smooth decision boundaries are needed
3. Small datasets make the tree unstable across different patient samples
4. Extrapolation beyond the training range is required

Controlling complexity through depth limits trades a tiny accuracy loss for much better generalization — exactly what the Health Department needs for a screening tool that must perform reliably across diverse hospital populations, lab equipment, and imaging protocols.

## 8. Wrap-Up: Key Takeaways

### What We Learned Today:

1. **Tree Mechanics**: Recursive partitioning with greedy splits — the tree asks "which feature at which threshold best separates the classes?" at every node, and stops when a depth or leaf-size constraint is hit.
2. **Overfitting Risk**: Unrestricted trees memorize perfectly, which is why training accuracy hits 1.00 while test accuracy does not. Always cap complexity.
3. **Complexity Control**: `max_depth`, `min_samples_split`, and `min_samples_leaf` are the three knobs that actually matter — tune them with CV, not by eyeballing.
4. **Interpretability**: A decision tree's path from root to leaf is a plain-English screening rule. That is the property logistic regression cannot match — and it is the reason trees stay in the toolkit even when ensembles beat them on raw AUC.
5. **Class Imbalance**: `class_weight='balanced'` is the first-resort tool when the minority class is being ignored by the default splits. SMOTE is not the default — reach for `class_weight` first.
6. **Limitations**: High variance across training sets, poor extrapolation beyond the training range, and a bias toward features with many unique values. These limitations are exactly what Random Forests (next notebook) are designed to fix.

### Critical Rules:

> **"Never use unrestricted trees in production."**

> **"Always tune depth with cross-validation — the CI-overlap rule from NB08 picks the depth."**

> **"Reach for `class_weight` before reaching for SMOTE."**

> **"Trees are building blocks for ensembles — every random forest is a crowd of the trees you built today."**

### Next Steps:

- **NB12 (Random Forests)** fixes the single tree's high-variance problem by averaging many trees fit on bootstrapped samples. If today's lesson was "one tree is unstable," NB12's lesson is "a hundred unstable trees together are stable." It also introduces the four-method feature-importance reconciliation table that guides every interpretation question in NB15.
- **NB13 (Gradient Boosting)** goes the other direction — sequentially correcting errors rather than averaging independent trees. Same building block, different ensemble philosophy, typically higher accuracy at the cost of more careful tuning.
- **Your final project:** every tree-based model you build for the project rests on today's depth-sweep and class-weight discipline. If your project has an imbalanced target, start with `class_weight='balanced'` before touching any resampling technique.

> **A question that often comes up here:** *"If Random Forests are strictly better than single trees, why did we spend a whole notebook on single trees?"* Because you cannot understand a forest without understanding a tree. Every bagging and boosting argument in the next two notebooks takes for granted that you know *why* a single tree overfits, *how* depth interacts with complexity, and *what* `class_weight` does. The decision tree is the building block; the rest of Week 3 is what you do with many of them.

---

## Participation Assignment Submission Instructions

### To Submit This Notebook:

1. **Complete all exercises**: Fill in both PAUSE-AND-DO exercise cells with your findings
2. **Run All Cells**: Execute `Runtime → Run all` to ensure everything works
3. **Save a Copy**: `File → Save a copy in Drive or Download the .ipynb extension`
4. **Submit**: Upload your `.ipynb` file in the participation assignment you find in the course Brightspace page.

### Before Submitting, Check:

- [ ] All cells execute without errors
- [ ] All outputs are visible
- [ ] Both exercise responses are complete
- [ ] Notebook is shared with correct permissions
- [ ] You can explain every line of code you wrote

### Next Step:

Complete the **Quiz** in Brightspace (auto-graded)

---

## Bibliography

- James, G., Witten, D., Hastie, T., & Tibshirani, R. (2021). *An Introduction to Statistical Learning with Python* - Tree-Based Methods (trees, pruning)
- Hastie, T., Tibshirani, R., & Friedman, J. (2009). *The Elements of Statistical Learning* - CART foundations and complexity control
- Breiman, L., Friedman, J., Stone, C. J., & Olshen, R. A. (1984). *Classification and Regression Trees*
- scikit-learn User Guide: [Decision Trees](https://scikit-learn.org/stable/modules/tree.html)

---



<center>

Thank you!

</center>